In [61]:
# Unigram
# Bigram
# Trigrma
# Laplace Smooting : 확률이 0되는문제 해결

In [62]:
from konlpy.tag import Okt
from nltk.util import ngrams
from collections import Counter, defaultdict
import pandas as pd

In [63]:
# url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
# df = pd.read_csv(url)
# df.head()

In [64]:
corpus = [
    "나는 오늘 맛있는 점심을 먹었다",
    "오늘 점심을 먹고 카페에 갔다",
    "나는 카페에 가서 맛있는 커피를 마셨다",
    "맛있는 점심을 먹는 것은 행복하다",
    "오늘 날씨는 정말 좋다",
    "나는 오늘 공부를 열심히 했다"
]

$P(w_n | w_{n-1}) = \frac{Count(w_{n-1}, w_n)}{Count(w_{n-1})}$

In [65]:
# Ngram모델  n값에 따라서 unigram bigram trigram모델 생성 통합함수
# 문장의 시작과 끝에 <s> </s>  확률 계산 MLE방식

In [66]:
def build_ngram_model(corpus, n):
    """
    n-gram 모델을 구축하여 조건부 확률을 계산합니다.
    """
    model = defaultdict(lambda: defaultdict(lambda: 0))
    okt = Okt()
    for sentence in corpus:
        tokens = sentence.split()  # 한국어 형태소 또는 어간 분리
        # tokens = [word for word, pos in okt.pos(sentence)]  # 형태소
        
        # 문장의 시작과 끝을 알리는 패딩 추가 (<s>: 시작, </s>: 끝)
        ngram_list = list(ngrams(tokens, n, pad_left=True, pad_right=True, 
                                 left_pad_symbol='<s>', right_pad_symbol='</s>'))
        
        for ngram in ngram_list:
            context = ngram[:-1]  # n-1개의 이전 단어들
            target = ngram[-1]   # 현재 단어
            model[context][target] += 1
            
    # 확률 계산 (MLE)
    for context in model:
        total_count = float(sum(model[context].values()))
        for target in model[context]:
            model[context][target] /= total_count
            
    return model

# 모델 생성
bigram_model = build_ngram_model(corpus, 2)
trigram_model = build_ngram_model(corpus, 3)

In [67]:
def check_probability(model, context, target):
    prob = model[tuple(context)].get(target, 0)
    print(f"P('{target}' | {context}) = {prob:.4f}")


print("--- Bigram Test ---")
check_probability(bigram_model, ['오늘'], '점심을')
check_probability(bigram_model, ['나는'], '오늘')

print("\n--- Trigram Test ---")
check_probability(trigram_model, ['맛있는', '점심을'], '먹었다')
check_probability(trigram_model, ['나는', '오늘'], '공부를')

--- Bigram Test ---
P('점심을' | ['오늘']) = 0.2500
P('오늘' | ['나는']) = 0.6667

--- Trigram Test ---
P('먹었다' | ['맛있는', '점심을']) = 0.5000
P('공부를' | ['나는', '오늘']) = 0.5000


In [68]:
def generate_next_word(model, context):
    context = tuple(context)
    if context not in model:
        return "<Unknown>"
    
    # 가장 높은 확률을 가진 단어 추출
    next_word = max(model[context].items(), key=lambda x: x[1])[0]
    return next_word

print(f"'카페에' 다음에 올 단어: {generate_next_word(bigram_model, ['카페에'])}")
print(f"'맛있는 점심을' 다음에 올 단어: {generate_next_word(trigram_model, ['맛있는', '점심을'])}")

'카페에' 다음에 올 단어: 갔다
'맛있는 점심을' 다음에 올 단어: 먹었다


In [69]:
# Laplace Smooting  희소단어가 나올때 분모가 0되는 현상 방지

def build_ngram_model_with_smoothing(corpus, n):
    model = defaultdict(lambda: defaultdict(lambda: 0))
    vocab = set()
    
    for sentence in corpus:
        tokens = sentence.split()
        vocab.update(tokens)
        ngram_list = list(ngrams(tokens, n, pad_left=True, pad_right=True, 
                                 left_pad_symbol='<s>', right_pad_symbol='</s>'))
        for ngram in ngram_list:
            model[ngram[:-1]][ngram[-1]] += 1
    
    V = len(vocab) + 1 # </s> 기호 포함  분모가 0이되는것을 방지하기위해서 분모에 추가
    
    # Laplace Smoothing 적용 확률 계산
    smoothed_model = defaultdict(lambda: defaultdict(lambda: 0))
    for context in model:
        total_count = sum(model[context].values())
        for target in model[context]:
            # (count + 1) / (total_count + V)
            smoothed_model[context][target] = (model[context][target] + 1) / (total_count + V)
            
    return smoothed_model

smoothed_bigram = build_ngram_model_with_smoothing(corpus, 2)
print("Smoothing 적용 후 '오늘' -> '커피를' (데이터에 없는 조합) 확률:")
print(f"{smoothed_bigram[('오늘',)].get('커피를', 1 / (sum(bigram_model[('오늘',)].values()) + 15)):.4f}")

Smoothing 적용 후 '오늘' -> '커피를' (데이터에 없는 조합) 확률:
0.0625


In [70]:
# 클래스별 모델 학습: 긍정(Positive)리뷰들로만 구성된 N-gram 모델과 부정리뷰들만 구성된 N-gram모델을 각각 만든다
# 확률계산 : 새로운 리뷰가 들어오면 리뷰가 긍정모델에서 확률과 부정모델에서 확률 각각 계산
# 두개 비교해서 높은 값을 가지는 클래스로 분류
# 여러단어들의 확률을 계속 곱하면 값이 0에 한없이 가까워지는 underflow현상 발생--> 확률에 log를 취해서 더해준다

In [71]:
data = {
    'reviews': [
        '이 커피 진짜 맛있다', '오늘 커피 향이 너무 좋아', '최고의 커피 추천합니다', # 긍정 (1)
        '커피가 너무 쓰다', '오늘 커피 맛이 별로네', '최악의 커피 절대 안 먹어'    # 부정 (0)
    ],
    'target': [1, 1, 1, 0, 0, 0]
}
df = pd.DataFrame(data)

N = 2 # Bigram 사용

# 클래스별로 코퍼스 분리 및 모델 학습
models_info = {}
for label in df['target'].unique():
    # 해당 클래스의 리뷰만 추출하여 코퍼스 생성
    corpus = df[df['target'] == label]['reviews'].tolist()
    # 모델 학습
    models_info[label] = build_ngram_model_with_smoothing(corpus, N)

print("모델 학습 완료!\n")

# 새로운 리뷰 테스트
test_reviews = [
    '오늘 커피 진짜 최고의 맛',
    '커피가 별로네 안 먹어'
]

for review in test_reviews:
    prediction = predict_sentiment(review, models_info, n=N)
    sentiment = "긍정" if prediction == 1 else "부정"
    print(f"리뷰: '{review}' -> 예측된 감성: {sentiment} (Class: {prediction})")

모델 학습 완료!



ValueError: too many values to unpack (expected 3)

## 정답

In [ ]:
import math
def build_ngram_model_with_smoothing(corpus, n):
    model = defaultdict(lambda: defaultdict(lambda: 0))
    vocab = set()
    
    for sentence in corpus:
        tokens = sentence.split()
        vocab.update(tokens)
        ngram_list = list(ngrams(tokens, n, pad_left=True, pad_right=True, 
                                 left_pad_symbol='<s>', right_pad_symbol='</s>'))
        for ngram in ngram_list:
            model[ngram[:-1]][ngram[-1]] += 1
            
    V = len(vocab) + 1 # </s> 기호 포함
    
    smoothed_model = defaultdict(lambda: defaultdict(lambda: 0))
    for context in model:
        total_count = sum(model[context].values())
        for target in model[context]:
            # (count + 1) / (total_count + V)
            smoothed_model[context][target] = (model[context][target] + 1) / (total_count + V)
            
    return smoothed_model, model, V


# 2. 문장의 로그 확률(Log Probability)을 계산하는 함수
def calculate_sentence_log_prob(sentence, smoothed_model, raw_model, V, n):
    tokens = sentence.split()
    ngram_list = list(ngrams(tokens, n, pad_left=True, pad_right=True, 
                             left_pad_symbol='<s>', right_pad_symbol='</s>'))
    
    log_prob = 0.0
    for ngram in ngram_list:
        context = ngram[:-1]
        target = ngram[-1]
        
        # 1. smoothed_model에 확률이 이미 계산되어 있는 경우
        if target in smoothed_model[context]:
            prob = smoothed_model[context][target]
        # 2. 컨텍스트는 아는데 해당 타겟 단어를 처음 보는 경우 (스무딩 적용)
        elif context in raw_model:
            total_count = sum(raw_model[context].values())
            prob = 1 / (total_count + V)
        # 3. 컨텍스트 자체를 처음 보는 경우 (스무딩 적용)
        else:
            prob = 1 / V
            
        log_prob += math.log(prob)
        
    return log_prob


# 3. 감성 예측 함수
def predict_sentiment(sentence, models_info, n=2):
    best_class = None
    max_log_prob = -float('inf')
    
    # 각 클래스(긍정, 부정 등)별로 확률을 계산하여 가장 높은 확률을 가진 클래스 선택
    for label, (smoothed_model, raw_model, V) in models_info.items():
        log_prob = calculate_sentence_log_prob(sentence, smoothed_model, raw_model, V, n)
        
        if log_prob > max_log_prob:
            max_log_prob = log_prob
            best_class = label
            
    return best_class